# User Experience para ciência de dados
Previsão de atrasos em pedidos.

In [187]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import pandas as pd

# Importar lógica de preparação de dados existente
from data_preparation_final import load_and_clean_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [188]:
def prepare_data_for_lgbm(df):
    """
    Limpeza e filtragem sugerida para o modelo LightGBM.
    """
    print("\n--- Preparação de Dados para LightGBM ---")
    inicial = len(df)
    
    # 1. Limpeza Crítica: dropna em campos fundamentais
    cols_limpeza = ['dt_despacho_pedido', 'dt_entrega_pedido', 'dt_pagamento_pedido', 'qtd_dias_tat']
    
    # --- OPÇÃO DE IMPUTAÇÃO (COMENTADA) ---
    # Se decidirmos não dropar os nulos de dt_pagamento_pedido (6.4%):
    # df['dt_pagamento_pedido'] = df['dt_pagamento_pedido'].fillna(method='ffill') # ou uma data fixa 'Pendente'
    # df['dias_aprovacao'] = df['dias_aprovacao'].fillna(-1) # Categoria para 'Pendente'
    # --------------------------------------
    
    df = df.dropna(subset=cols_limpeza).copy()
    
    posterior = len(df)
    print(f"Registros antes: {inicial}")
    print(f"Registros após limpeza (dropna): {posterior}")
    print(f"Perda de dados: {1 - (posterior/inicial):.2%}")

    # 2. Tratamento de Outliers (Percentil 99 em qtd_dias_tat)
    limite_99 = df['qtd_dias_tat'].quantile(0.99)
    df = df[df['qtd_dias_tat'] <= limite_99].copy()
    print(f"Outliers removidos (TAT > {limite_99:.1f} dias): {posterior - len(df)}")

    return df

## Limpeza e tratamento de dados

In [189]:
# 1. Carregar e Limpar
input_file = "pedidos_logistica.parquet"
df = load_and_clean_data(input_file, drop_ids=False)

if df is not None:
    df = prepare_data_for_lgbm(df)

Carregando dados de pedidos_logistica.parquet...
Colunas após renomeação:
id, cod_pedido, uf
grp_transportadora, dt_despacho_pedido, dt_entrega_pedido
dt_previsao_entrega_cliente, dt_criacao, dt_pagamento_pedido
flg_existem_ocorrencias, tp_praca, des_unidade_negocio
des_cd_origem, qtd_dias_tat, tp_performance_entrega
cidade_destinatario
Removendo registros inconsistentes (entrega antecede despacho): 7
Realizando engenharia de features...
Processamento concluído. Formato final: (490210, 23)
Salvando dataset limpo em pedidos_logistica_limpo.parquet...

--- Preparação de Dados para LightGBM ---
Registros antes: 490210
Registros após limpeza (dropna): 458687
Perda de dados: 6.43%
Outliers removidos (TAT > 14.0 dias): 3596


In [190]:
# Manter para ter coerencia com 'trabalho' - Pedro
df = df.drop(
    columns=[
        # 'row_id',
        # 'hr_despacho_pedido',
        'dt_entrega_pedido',
        # 'hr_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
 )

df.head(10)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,...,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,hr_despacho_pedido,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,is_atrasado,dias_restantes_prazo
0,67044,127510252-1,RJ,Transportadora 3,2023-11-27 14:57:25,2023-12-06,2023-11-24,2023-11-24,Capital,Multi,...,5.0,1,RIO DE JANEIRO,14,3.623206,4.209201,-4.167593,7.832407,0,12.0
1,67045,122170353-1,SE,Transportadora 3,2023-07-19 11:41:44,2023-07-27,2023-07-17,2023-07-17,Capital,Mono,...,6.0,1,ARACAJU,11,2.487315,6.127512,-1.385174,8.614826,0,10.0
2,67046,125676313-1,RN,Transportadora 2,2023-11-08 15:44:19,2023-11-20,2023-11-07,2023-11-07,Interior,Mono,...,7.0,1,NATAL,15,1.655775,9.878773,-1.465451,11.534549,0,13.0
3,67047,124809810-1,PA,Transportadora 2,2023-10-16 12:40:54,2023-10-25,2023-10-14,2023-10-14,Reg. Metropolitana,Multi,...,6.0,1,ANANINDEUA,12,2.528403,6.918843,-1.552755,9.447245,0,11.0
5,67049,124134539,MG,Transportadora 1,2023-09-25 08:47:42,2023-10-02,2023-09-24,2023-09-24,Interior,Multi,...,4.0,1,JUIZ DE FORA,8,1.366458,3.065231,-3.568310,4.431690,0,8.0
6,67050,123148738-1,MG,Transportadora 1,2023-08-22 14:13:44,2023-08-30,2023-08-21,2023-08-21,Interior,Multi,...,5.0,1,BAEPENDI,14,1.592870,6.010185,-1.396944,7.603056,0,9.0
7,67051,127085176-1,TO,Transportadora 2,2023-11-24 22:14:31,2023-12-06,2023-11-22,2023-11-22,Capital,Mono,...,6.0,1,PALMAS,22,2.926748,5.557639,-5.515613,8.484387,0,14.0
8,67052,122966837-3,RJ,Transportadora 1,2023-08-16 14:53:27,2023-08-21,2023-08-15,2023-08-15,Reg. Metropolitana,Multi,...,3.0,1,NITEROI,14,1.620451,2.031829,-2.347720,3.652280,0,6.0
9,67053,122721965-1,MG,Transportadora 1,2023-08-07 09:41:30,2023-08-11,2023-08-07,2023-08-06,Interior,Multi,...,3.0,1,POCOS DE CALDAS,9,1.403819,2.162836,-1.433345,2.566655,0,5.0
10,67054,126966272,AC,Transportadora 1,2023-11-23 01:13:53,2023-12-11,2023-11-21,2023-11-21,Capital,Multi,...,9.0,1,RIO BRANCO,1,2.051308,11.681597,-6.267095,13.732905,0,20.0


## Profiling de dados

In [191]:
# profile = ProfileReport(df, title="Profiling Report")
# html = profile.to_html()
# output_file = 'report.html'
# with open(output_file, 'w') as f:
#     f.write(html)

In [192]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# TODO adaptar modelo abaixo com essa funcao
def train_lgbm(df):
    """
    Treinamento do modelo LightGBM usando a API nativa e recursos de classificação.
    """
    # 1. Divisão Temporal 70/30
    df = df.sort_values('dt_criacao')
    split_idx = int(len(df) * 0.7)
    train_test_data = df.iloc[:split_idx].copy()
    holdout_data = df.iloc[split_idx:].copy()    
    
    # 2. Definição de Features e Alvo
    # Categóricas (Nativas do LightGBM)
    cat_features = [
        'cidade_destinatario', 
        'uf', 
        'grp_transportadora', 
        'dt_pagamento_pedido',
        'dt_previsao_entrega_cliente',
        'dt_criacao',
        'tp_praca', 
        'des_unidade_negocio', 
        'des_cd_origem'
    ]
    # Numéricas
    # num_features = ['dias_aprovacao']
    num_features = []
    
    target = 'tp_performance_entrega'
    
    # Converter categóricas para o tipo 'category' do Pandas (Exigência do LightGBM)
    for col in cat_features:
        train_test_data[col] = train_test_data[col].astype('category')
        holdout_data[col] = holdout_data[col].astype('category')
    
    features = cat_features + num_features

    X = train_test_data[features]
    y = train_test_data[target]
    
    # Split para Treino e Validação (dentro dos 70%)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Cálculo do scale_pos_weight para desbalanceamento
    # Justificativa: LightGBM lida melhor com classes minoritárias se aumentarmos o peso dos positivos (atrasos).
    pos_count = y_train.sum()
    neg_count = len(y_train) - pos_count
    spw = neg_count / pos_count
    
    print(f"\nConfigurando scale_pos_weight: {spw:.2f} (Classe 'Atrasado' é minoritária)")

    # 4. Configuração do Modelo
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'scale_pos_weight': spw,
        'learning_rate': 0.05,
        'num_leaves': 128,
        'max_depth': -1,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'seed': 42
    }

    print("Iniciando treinamento com LightGBM...")
    
    train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set, categorical_feature=cat_features)
    
    model = lgb.train(
        params,
        train_set,
        valid_sets=[train_set, val_set],
        num_boost_round=1000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50)
        ]
    )

    return model, holdout_data, features

In [193]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ---------------------------------------------------------
# Step 3: Split Features (X) and Target (y)
# ---------------------------------------------------------
selected_features = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'dt_previsao_entrega_cliente',
    'dt_criacao',
    'dt_pagamento_pedido',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
]

# 1. Divisão Temporal 70/30
df = df.sort_values('dt_criacao')
split_idx = int(len(df) * 0.7)
# df de treinamento e avaliação do modelo
train_test_data = df.iloc[:split_idx].copy()
# df para simulação (dados não conhecidos pelo modelo)
holdout_data = df.iloc[split_idx:].copy()

available_features = [col for col in selected_features if col in train_test_data.columns]
missing_features = [col for col in selected_features if col not in train_test_data.columns]

if missing_features:
    print('Missing columns (ignored):', missing_features)

X = train_test_data[available_features].copy()
y = train_test_data['tp_performance_entrega']

# Remove rows with missing target
valid_mask = y.notna()
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].astype('int32').copy()

# Convert requested date columns to numeric representation (ordinal days)
date_cols = ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
for col in [c for c in date_cols if c in X.columns]:
    X[col] = pd.to_datetime(X[col], errors='coerce')
    X[col] = X[col].map(lambda x: x.toordinal() if pd.notna(x) else np.nan).astype('float32')
print(X[col])

# Encode requested categorical columns as numeric codes
categorical_cols = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
]
for col in [c for c in categorical_cols if c in X.columns]:
    X[col] = X[col].astype('category').cat.codes.replace(-1, np.nan).astype('float32')

# LightGBM does not accept object/datetime columns directly
unsupported_cols = X.select_dtypes(include=['object', 'datetime64[ns]', 'datetimetz']).columns
if len(unsupported_cols) > 0:
    print('Dropping unsupported columns:', list(unsupported_cols))
X = X.drop(columns=unsupported_cols, errors='ignore')

print('Training columns:', list(X.columns))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Step 4: Initialize and Train the Model
# ---------------------------------------------------------
# model = lgb.LGBMClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=5,
#     random_state=42,
#     is_unbalance=True
# )
# Sobre a causa das mensagens " LightGBM Warning No further splits with positive gain, best gain: -inf"
# Com mais de 400 mil registros, o problema certamente não é falta de dados (o min_child_samples=20 padrão seria atendido facilmente). 
# O aviso aparece porque você está limitando muito o modelo com max_depth=5 em um cenário de logística que parece ser complexo.
# Com mais de 400k linhas, uma profundidade de 5 níveis (máximo de 32 folhas) é muito pouco para capturar as nuances de logística 
# (como variações por cidade ou transportadora). O LightGBM tenta criar divisões, mas como ele já atingiu o limite de profundidade ou 
# as combinações restantes não batem com o is_unbalance=True, ele desiste e gera o aviso.

# Sugestão: Deixe o modelo crescer mais e controle pelo número de folhas, que é mais eficiente no LightGBM.
model = lgb.LGBMClassifier(
    n_estimators=200,      # Aumente um pouco já que tem muitos dados
    learning_rate=0.05,    # Reduza a taxa para aprender com mais calma
    num_leaves=63,         # Aumente a complexidade (2^max_depth - 1)
    max_depth=-1,          # Deixe o crescimento livre (controlado por num_leaves)
    random_state=42,
    is_unbalance=True,
    verbose=-1             # Silencie o aviso agora que ajustamos a estrutura
)

# Fit model
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Step 5: Make Predictions (The "Risk Score")
# ---------------------------------------------------------
predictions_binary = model.predict(X_test)
predictions_proba = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Step 6: Evaluate Model Quality
# ---------------------------------------------------------
accuracy = accuracy_score(y_test, predictions_binary)
precision = precision_score(y_test, predictions_binary, zero_division=0)
recall = recall_score(y_test, predictions_binary, zero_division=0)
f1 = f1_score(y_test, predictions_binary, zero_division=0)
roc_auc = roc_auc_score(y_test, predictions_proba)
cm = confusion_matrix(y_test, predictions_binary)

print('=== Model Metrics ===')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, predictions_binary, zero_division=0))

296320    738702.0
388636    738702.0
209761    738702.0
233401    738702.0
57140     738701.0
            ...   
9371      738838.0
304768    738838.0
497635    738838.0
484979    738838.0
286153    738841.0
Name: dt_pagamento_pedido, Length: 318563, dtype: float32
Training columns: ['cidade_destinatario', 'uf', 'grp_transportadora', 'dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem']
=== Model Metrics ===
Accuracy : 0.8275
Precision: 0.9864
Recall   : 0.8331
F1-score : 0.9033
ROC-AUC  : 0.8275

Confusion Matrix:
[[ 1415   708]
 [10280 51310]]

Classification Report:
              precision    recall  f1-score   support

           0       0.12      0.67      0.20      2123
           1       0.99      0.83      0.90     61590

    accuracy                           0.83     63713
   macro avg       0.55      0.75      0.55     63713
weighted avg       0.96      0.83      0.88     63713



In [194]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(10, 6))
# lgb.plot_importance(model, ax=ax)
# plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
# plt.close(fig)
# print('Saved: feature_importance.png')

In [195]:
# ---------------------------------------------------------
# Step 7: View the Results
# ---------------------------------------------------------
results_df = X_test.copy()
results_df['probabilidade_atraso'] = predictions_proba
results_df['risco_semaforo'] = pd.cut(
    results_df['probabilidade_atraso'],
    bins=[-0.1, 0.3, 0.7, 1.1],
    labels=['🟢 Verde', '🟡 Amarelo', '🔴 Vermelho']
)
print(results_df[['probabilidade_atraso', 'risco_semaforo']].head(20))

        probabilidade_atraso risco_semaforo
247532              0.632485      🟡 Amarelo
123175              0.805893     🔴 Vermelho
245490              0.848686     🔴 Vermelho
353735              0.253816        🟢 Verde
254993              0.756717     🔴 Vermelho
274410              0.778151     🔴 Vermelho
368085              0.671396      🟡 Amarelo
140789              0.679579      🟡 Amarelo
238961              0.765998     🔴 Vermelho
77209               0.408000      🟡 Amarelo
377229              0.744341     🔴 Vermelho
148611              0.997447     🔴 Vermelho
502849              0.581380      🟡 Amarelo
174022              0.928081     🔴 Vermelho
113941              0.689932      🟡 Amarelo
146491              0.853495     🔴 Vermelho
420833              0.276191        🟢 Verde
386558              0.678407      🟡 Amarelo
235289              0.784788     🔴 Vermelho
375006              0.515200      🟡 Amarelo


In [180]:
import joblib

# Build categorical mappings from training data
categorical_mappings = {}
for col in [c for c in categorical_cols if c in train_test_data.columns]:
    cats = pd.Series(train_test_data.loc[valid_mask, col].astype("string").dropna().unique()).sort_values().tolist()
    categorical_mappings[col] = {v: i for i, v in enumerate(cats)}

bundle = {
    "model": model,
    "selected_features": selected_features,
    "date_cols": ["dt_previsao_entrega_cliente", "dt_criacao", "dt_pagamento_pedido"],
    "categorical_cols": [c for c in categorical_cols if c in selected_features],
    "categorical_mappings": categorical_mappings,
    "threshold": 0.5
}

joblib.dump(bundle, "model_bundle.joblib")
print("Saved model_bundle.joblib")

Saved model_bundle.joblib


In [182]:
train_test_data[train_test_data['tp_performance_entrega'] == 0]
# len(train_test_data)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,...,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,hr_despacho_pedido,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,is_atrasado,dias_restantes_prazo
420792,489065,121525499-1,GO,Transportadora 4,2023-07-01 10:20:27,2023-06-30,2023-07-01,2023-06-25,Capital,Multi,...,7.0,0,GOIANIA,10,6.430868,3.317037,4.747905,3.747905,1,5.0
420241,488506,121666681-1,BA,Transportadora 2,2023-07-03 20:33:48,2023-07-10,2023-07-01,2023-07-01,Capital,Mono,...,7.0,0,SALVADOR,20,2.856806,8.102847,1.959653,10.959653,1,9.0
428520,496872,121669260-1,MA,Transportadora 2,2023-07-03 20:47:07,2023-07-13,2023-07-01,2023-07-01,Capital,Mono,...,12.0,0,SAO LUIS,20,2.866053,14.751435,5.617488,17.617488,1,12.0
415771,483999,121674017-1,RJ,Transportadora 1,2023-07-03 14:01:46,2023-07-06,2023-07-01,2023-07-01,Capital,Mono,...,6.0,0,RIO DE JANEIRO,14,2.584560,7.075139,4.659699,9.659699,1,5.0
420398,488663,121671849-1,PB,Transportadora 3,2023-07-03 16:29:56,2023-07-10,2023-07-01,2023-07-02,Capital,Mono,...,7.0,0,JOAO PESSOA,16,1.687454,7.675521,1.362975,10.362975,1,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417891,486131,126206866-1,GO,Transportadora 2,2023-11-18 15:30:19,2023-11-27,2023-11-14,2023-11-14,Interior,Multi,...,9.0,0,APARECIDA DE GOIANIA,15,4.646053,9.784120,1.430174,14.430174,1,13.0
426375,494707,126187628,SP,Transportadora 8,2023-11-16 18:40:30,2023-11-17,2023-11-14,2023-11-14,Capital,Multi,...,2.0,0,SAO PAULO,18,2.778125,2.755220,2.533345,5.533345,1,3.0
433500,501902,126248826,RJ,Transportadora 3,2023-11-15 23:56:30,2023-11-22,2023-11-14,2023-11-14,Capital,Multi,...,7.0,0,RIO DE JANEIRO,23,1.997569,8.664456,2.662025,10.662025,1,8.0
426115,494445,126257961-1,SP,Transportadora 1,2023-11-17 19:21:07,2023-11-21,2023-11-14,2023-11-14,Capital,Multi,...,5.0,0,SAO PAULO,19,3.806331,5.060868,1.867199,8.867199,1,7.0
